# Exercise 5: Sinusoidal model

This exercise explores sinusoidal analysis and tracking using the sinusoidal model from `smstools`. It has four parts:
1. Minimise the frequency estimation error of a sinusoid
2. Track a two-component linear chirp
3. Track sinusoids of very different amplitudes
4. Sinusoidal modelling of a multicomponent signal

### Relevant concepts

**Chirp signals:** A chirp is a signal whose instantaneous frequency changes over time. A *linear* chirp sweeps frequency at a constant rate. Analysing a chirp with the STFT involves a fundamental tradeoff: resolving two closely spaced frequency components requires a *long* window (narrow main lobe), but tracking a rapidly changing frequency requires a *short* window (good time localisation). The optimal window size balances these competing demands.

**Sinusoidal modelling and sine tracking:** The sinusoidal model represents each audio frame as a sum of sinusoids. The `sineModelAnal()` function detects spectral peaks in each frame and links them across frames into *tracks*. The key parameters are:

- `maxnSines`: maximum number of sinusoidal tracks per frame
- `minSineDur`: minimum track duration in seconds (shorter tracks are discarded as spurious)
- `freqDevOffset`: minimum allowed frequency deviation at 0 Hz (sets the tracking tolerance)
- `freqDevSlope`: how much the deviation tolerance grows with frequency (compensates for the tendency of higher-frequency tracks to vary more)

Setting `minSineDur = 0` retains all detected peaks with no duration filtering.

**Tracking low-amplitude sinusoids:** When two sinusoids have very different amplitudes, the *sidelobes* of the stronger component can mask the main lobe of the weaker one. The choice of analysis window is critical: windows with high sidelobe levels (e.g. rectangular) will bury weak components, while windows with strong sidelobe attenuation (e.g. Blackman-Harris) keep sidelobes well below the weak component's main lobe.


## Part 1 – Minimising the frequency estimation error of a sinusoid

The function `min_freq_est_err()` should find the smallest valid window size `M` such that the frequency estimation error is less than 0.05 Hz, and return the estimated frequency along with the parameters used.

**Analysis setup:**
- Window type: Blackman
- Analysis frame: centred at $t = 0.5$ s
- Peak detection threshold: $t = -40$ dB
- Use `dftAnal()`, `peakDetection()`, and `peakInterp()` from `dftModel.py`

**Finding M:** Iterate over candidate window sizes of the form $M = 100k + 1$ for $k = 1, 2, 3, \ldots$ and stop at the first value for which $|f - f_{\mathrm{est}}| < 0.05$ Hz. For each `M`, set the FFT size `N` to the smallest power of 2 larger than `M`.

**Inputs:**
- `input_file` (str): path to a `.wav` file containing a single stationary sinusoid (length ≥ 1 s)
- `f` (float): true frequency of the sinusoid in Hz (range: 100–2000 Hz)

**Output:** tuple `(f_est, M, N)` — estimated frequency in Hz, window size, and FFT size.


In [2]:
import numpy as np
from scipy.signal import get_window
import math
from smstools.models import dftModel as DFT
from smstools.models import utilFunctions as UF
from smstools.models import stft
from smstools.models import sineModel as SM
import IPython.display as ipd

In [3]:
# E5 - 1.1: Complete the function min_freq_est_err()

def min_freq_est_err(input_file, f):
    """Best estimate the frequency of a sinusoid by iterating over different sizes of analysis window.

    Args:
            input_file (str): wav file
            f (float): frequency of the sinusoid present in the input audio signal (Hz)

    Result:
            f_est (float): estimated frequency of the sinusoid (Hz)
            M (int): Window size
            N (int): FFT size

    """
    # analysis parameters:
    window = 'blackman'
    t = -40

    ### Your code here


Test cases for `min_freq_est_err()`:

**Test case 1:** `sine-490.wav`, `f = 490.0 Hz` → `M = 1101`, `N = 2048`, `f_est ≈ 489.963 Hz`, error ≈ 0.037 Hz

**Test case 2:** `sine-1000.wav`, `f = 1000.0 Hz` → `M = 1101`, `N = 2048`, `f_est ≈ 1000.02 Hz`, error ≈ 0.020 Hz

**Test case 3:** `sine-200.wav`, `f = 200.0 Hz` → `M = 1201`, `N = 2048`, `f_est ≈ 200.038 Hz`, error ≈ 0.038 Hz

For each test case, generate a new sinusoid at `f_est` and listen to it alongside the original to check whether the estimation error is perceptually audible.


In [ ]:
# E5 - 1.2: For each test case, call min_freq_est_err(), generate and play both the original and the
#            estimated sinusoid, and compare them perceptually.

### Your code here


**E5 - 1.3: Conceptual questions (answer here)**

1. Why does a larger window size generally reduce the frequency estimation error? What property of the DFT is responsible? Verify by running `min_freq_est_err` on one test case and printing the error at each candidate `M`.
2. For the test cases above, the optimal `M` is around 1100–1200 samples. Does the required `M` depend on the frequency `f`? Try `f = 300 Hz` and `f = 1500 Hz` and compare the optimal `M` values. Explain the trend (or lack of one).
3. Can you hear a difference between the original and estimated sinusoids for any of the three test cases? What does this tell you about the perceptual relevance of a 0.05 Hz estimation error?


## Part 2 – Tracking a two-component linear chirp

Use `freq_tracker_error()` to find the window size `M` that best tracks both frequency components of `chirp-150-190-linear.wav`, aiming for a mean estimation error below 2 Hz on each component.

The file contains a two-component linear chirp: one sinusoid sweeping from 150 Hz to 1400 Hz and another from 190 Hz to 1440 Hz, both over 2 seconds. The two components are only 40 Hz apart throughout, so resolving them requires a large window — but a very large window will smear the rapidly changing frequencies. Finding the right balance is the core challenge.

**Fixed parameters:** `window = 'blackman'`, `t = -80`, `H = 128`.

**Your task:** Choose `M` analytically (think about the frequency separation and the required main-lobe width), then verify with `freq_tracker_error()`. Plot the true vs. estimated frequency tracks for both your initial choice (`M = 1023` as a baseline) and your improved value.

> Tip: use `gen_time_stamps()` to generate the time axis and `gen_true_freq_tracks_chirp_150_190()` to get the reference tracks for comparison.


In [5]:
# functions used in exercises of Part 2 and 3

def gen_time_stamps(xlen, M, fs, H):
    """Generate frame time stamps for a given signal length and sampling rate.

    Args:
        xlen (int): duration of signal in samples
        M (int): window size
        fs (int): sampling rate
        H (int): hop size

    Result:
        np.array: time stamps

    """
    hM1 = int(np.floor((M+1)/2))
    hM2 = int(np.floor(M/2))
    xlen = xlen + 2*hM2
    pin = hM1
    pend = xlen - hM1
    tStamps = np.arange(pin,pend,H)/float(fs)
    return tStamps

def gen_true_freq_tracks_chirp_150_190(tStamps):
    """Generate the frequency values present in file "../sounds/chirp-150-190-linear.wav"

    Args:
        tStamps (np.array): time stamps

    Result:
        np.array: time stamps and frequency values of predefined chirp

    """
    fTrack = np.zeros((len(tStamps),2))
    fTrack[:,0] = np.transpose(np.linspace(190, 190+1250, len(tStamps)))
    fTrack[:,1] = np.transpose(np.linspace(150, 150+1250, len(tStamps)))
    return fTrack

def gen_true_freq_tracks_440_602(tStamps):
    """Generate the frequency values present in file "../sounds/sines-440-602-hRange.wav"

    Args:
        tStamps (np.array): time stamps

    Result:
        np.array: time stamps and frequency values of predefined chirp

    """
    fTrack = np.zeros((len(tStamps),2))
    fTrack[:,0] = np.transpose(440*np.ones((len(tStamps),1)))
    fTrack[:,1] = np.transpose(602*np.ones((len(tStamps),1)))
    return fTrack

def freq_tracker_error(input_file, fTrackTrue, window, t, H, M):
    """Estimate sinusoidal values of a sound

    Args:
        input_file (str): wav file including the path
        fTrackTrue (np.array): array of true frequency values, one row per time frame, one column per component
        window (str): window type used for analysis
        t (float): peak picking threshold (negative dB)
        H (int): hop size in samples
        M (int): window size in samples

   Result:
           float: mean estimation error
           np.array: estimated frequency values, one row per time frame, one column per component

    """

    N = int(pow(2, np.ceil(np.log2(M))))        # FFT Size, power of 2 larger than M
    maxnSines = 2                               # Maximum number of sinusoids at any time frame
    minSineDur = 0.0                            # minimum duration set to zero to not do tracking
    freqDevOffset = 30                          # minimum frequency deviation at 0Hz
    freqDevSlope = 0.001                        # slope increase of minimum frequency deviation

    fs, x = UF.wavread(input_file)              # read input sound
    w = get_window(window, M)                   # Compute analysis window
    # analyze the sound with the sinusoidal model
    fTrackEst, mTrackEst, pTrackEst = SM.sineModelAnal(x, fs, w, N, H, t, maxnSines, minSineDur, freqDevOffset, freqDevSlope)
    tailF = 20
    # Compute mean estimation error. 20 frames at the beginning and end not used to compute error
    meanErr = np.mean(np.abs(fTrackTrue[tailF:-tailF,:] - fTrackEst[tailF:-tailF,:]),axis=0)

    return (meanErr, fTrackEst)

**Baseline test case:** `M = 1023` gives mean errors of `[13.7, 528.5]` Hz — far above the 2 Hz target. The second component is almost completely untracked.

Start by reasoning about **why** `M = 1023` fails: what is the main-lobe width in Hz for that window size at the sampling rate of the file? Is it narrow enough to resolve components 40 Hz apart? Then calculate the minimum `M` theoretically, test it, and refine.


In [ ]:
# E5 - 2.1: Call freq_tracker_error() with the baseline M=1023 and plot true vs estimated tracks.
#            Then find and apply an improved M, and plot the new estimated tracks.

import matplotlib.pyplot as plt

H = 128
window = 'blackman'
t = -80
input_file = '../sounds/chirp-150-190-linear.wav'
fs, x = UF.wavread(input_file)

# baseline test
M = 1023
tStamps = gen_time_stamps(x.size, M, fs, H)
fTrackTrue = gen_true_freq_tracks_chirp_150_190(tStamps)

### Your code here


**E5 - 2.3: Conceptual questions (answer here)**

1. What is the Blackman window's main-lobe width (in Hz) for `M = 1023` at the sampling rate of the chirp file? Is it narrow enough to resolve two components 40 Hz apart? Repeat the calculation for your improved `M`.
2. Increasing `M` improves frequency resolution but degrades time resolution. From your plot of the estimated frequency tracks, do you observe any blurring or smearing of the chirp trajectory for large `M`? At what window size does this become visible?
3. The first and last 20 frames (`tailF = 20`) are excluded from the error calculation. Why? What happens to the frequency tracks in those boundary regions?


## Part 3 – Tracking sinusoids of different amplitudes

Use `freq_tracker_error()` to find the window type `window` and magnitude threshold `t` that best track both sinusoids in `sines-440-602-hRange.wav`, aiming for a mean estimation error below 2 Hz on each component.

The signal is $s = \sin(2\pi \cdot 440 \cdot t) + 2\times10^{-3}\sin(2\pi \cdot 602 \cdot t)$: one component at 440 Hz and a second at 602 Hz that is **1000 times weaker** (−60 dB). The weak component's main lobe must be detectable above the sidelobes of the dominant component.

**Fixed parameters:** `M = 2047`, `N = 4096`, `H = 128`, `minSineDur = 0.02`, `freqDevOffset = 10`, `freqDevSlope = 0.001`, `maxnSines = 2`.

**Your task:** Choose `window` based on its sidelobe attenuation (compare `'boxcar'`, `'hanning'`, `'hamming'`, `'blackman'`, `'blackmanharris'`), and set `t` (in dB, negative) to capture the weak component without picking up noise. Plot the true vs. estimated tracks for the baseline (`window='hanning'`, `t=-80`) and your improved parameters.

**Baseline test case:** `window = 'hanning'`, `t = -80` → mean errors `[0.196, 29.5]` Hz. The weak 602 Hz component is tracked poorly.


In [ ]:
# E5 - 3.1: Call freq_tracker_error() with the baseline parameters and plot true vs estimated tracks.
#            Then choose the best window and threshold, and plot the improved results.

M = 2047
N = 4096
H = 128
input_file = '../sounds/sines-440-602-hRange.wav'
fs, x = UF.wavread(input_file)
tStamps = gen_time_stamps(x.size, M, fs, H)
fTrackTrue = gen_true_freq_tracks_440_602(tStamps)

### Your code here


**E5 - 3.3: Conceptual questions (answer here)**

1. Look up (or compute) the sidelobe attenuation of the Hanning and Blackman-Harris windows. The weak sinusoid is −60 dB relative to the dominant one. Which window types have sidelobe levels low enough to keep the weak component's main lobe visible? Confirm with your experimental results.
2. Why does lowering the magnitude threshold `t` help track the weak component? What is the risk of setting `t` too low (e.g., `t = -120 dB`)?
3. With `window = 'hanning'` and `t = -80`, the 602 Hz component has error ≈ 30 Hz. From the frequency-track plot, does the tracker follow a sidelobe of the 440 Hz sinusoid instead of the 602 Hz main lobe? How can you tell?


## Part 4 – Sinusoidal modelling of a multicomponent signal

Perform the best possible sinusoidal analysis of `multiSines.wav` by exploring the `sineModelAnal()` parameters. The goal is a resynthesised sound that is perceptually indistinguishable from the input.

`multiSines.wav` combines several challenging features simultaneously:
- Sharp amplitude attacks
- Closely spaced frequency components across a wide amplitude range
- Time-varying chirp tracks that cross over in frequency

Listen to the sound and examine its spectrogram (using `models_GUI.py` or Sonic Visualizer) before choosing parameters.

**Parameters to set:** `window`, `M`, `N`, `t`, `minSineDur`, `maxnSines`, `freqDevOffset`, `freqDevSlope`.

The synthesis parameters `Ns = 512` and `H = 128` are fixed — do not modify them.

> This is an open-ended exploration. Start with a reasonable set of parameters, listen to the output, examine the frequency tracks, and iterate.


In [ ]:
# E5 - 4.1: 
# Set the analysis parameters of sineModelAnal() to perform the best analysis of multiSines.wav

import IPython.display as ipd

input_file = '../sounds/multiSines.wav'

### set the analysis parameters

window = 'XX'
M = XX
N = XX
t = XX
minSineDur = XX
maxnSines = XX
freqDevOffset = XX
freqDevSlope = XX


# no need to modify the code after here
Ns = 512                                      # size of fft used in synthesis
H = 128                                       # hop size (has to be 1/4 of Ns)

fs, x = UF.wavread(input_file)                # read input sound
w = get_window(window, M)                     # compute analysis window

# analyze the sound with the sinusoidal model
tfreq, tmag, tphase = SM.sineModelAnal(x, fs, w, N, H, t, maxnSines, minSineDur, freqDevOffset, freqDevSlope)

# synthesize the output sound from the sinusoidal representation
y = SM.sineModelSynth(tfreq, tmag, tphase, Ns, H, fs)

# create figure to show plots
plt.figure(figsize=(15, 9))

# frequency range to plot
maxplotfreq = 5000.0

# plot the input sound
plt.subplot(3,1,1)
plt.plot(np.arange(x.size)/float(fs), x)
plt.axis([0, x.size/float(fs), min(x), max(x)])
plt.ylabel('amplitude')
plt.xlabel('time (sec)')
plt.title('input sound: x')

# plot the sinusoidal frequencies
plt.subplot(3,1,2)
if (tfreq.shape[1] > 0):
    numFrames = tfreq.shape[0]
    frmTime = H*np.arange(numFrames)/float(fs)
    tfreq[tfreq<=0] = np.nan
    plt.plot(frmTime, tfreq)
    plt.axis([0, x.size/float(fs), 0, maxplotfreq])
    plt.title('frequencies of sinusoidal tracks')

# plot the output sound
plt.subplot(3,1,3)
plt.plot(np.arange(y.size)/float(fs), y)
plt.axis([0, y.size/float(fs), min(y), max(y)])
plt.ylabel('amplitude')
plt.xlabel('time (sec)')
plt.title('output sound: y')

ipd.display(ipd.Audio(data=x, rate=fs))
ipd.display(ipd.Audio(data=y, rate=fs))

**E5 - 4.3: Conceptual questions (answer here)**

1. Describe the main challenges you identified in `multiSines.wav` from the spectrogram. Which feature was hardest to handle — the sharp attacks, the wide amplitude range, or the crossing chirp tracks? Explain why.
2. Report the final parameter values you chose (`window`, `M`, `t`, `maxnSines`, `minSineDur`, `freqDevOffset`, `freqDevSlope`). For each parameter, explain *why* you chose that value, linking it to a specific property of the signal.
3. Listen carefully to the resynthesised output. Are there any audible artefacts or missing components? If so, describe them and explain which aspect of the sinusoidal model is responsible.
4. Is there a single parameter set that handles all the challenging features of this signal simultaneously, or did you have to make tradeoffs? Describe any tradeoffs you encountered.
